# DATATHON 2026 - Phase 1 MCQ Solver (DuckDB)

Notebook nay giai 10 cau MCQ bang SQL tren DuckDB theo huong reproducible va minh bach.

**Vi sao DuckDB?**
- Toi uu RAM: khong can load tat ca CSV vao Pandas cung luc.
- SQL manh va nhanh tren nhieu bang du lieu giao dich.
- De audit, de truy vet, de trinh bay voi giam khao theo kieu data-backed decision.

In [3]:
!pip install duckdb pandas ipython
import os
import glob
import duckdb
import pandas as pd
from IPython.display import display

con = duckdb.connect('gridbreaker.duckdb')

candidate_dirs = ['data_datathon', 'Data']
data_dir = next((d for d in candidate_dirs if os.path.isdir(d)), None)
if data_dir is None:
    raise FileNotFoundError('Khong tim thay thu muc du lieu. Vui long tao data_datathon/ hoac Data/.')

csv_files = sorted(glob.glob(os.path.join(data_dir, '*.csv')))
if not csv_files:
    raise FileNotFoundError(f'Khong tim thay file CSV trong {data_dir}')

for csv_path in csv_files:
    table_name = os.path.splitext(os.path.basename(csv_path))[0].lower()
    con.execute(
        f'''
        CREATE OR REPLACE TABLE "{table_name}" AS
        SELECT * FROM read_csv_auto(?, header=true, sample_size=-1)
        '''
        , [csv_path]
    )
    print(f'Loaded: {os.path.basename(csv_path)} -> table {table_name}')

print(f'Ingestion done: {len(csv_files)} tables loaded from {data_dir}')

  Using cached numpy-2.4.4-cp313-cp313-win_amd64.whl.metadata (6.6 kB)
   ---------------------------------------- 0.0/9.7 MB ? eta -:--:--
   - -------------------------------------- 0.3/9.7 MB ? eta -:--:--
   -------- ------------------------------- 2.1/9.7 MB 7.4 MB/s eta 0:00:02
   --------------- ------------------------ 3.7/9.7 MB 7.8 MB/s eta 0:00:01
   ---------------------- ----------------- 5.5/9.7 MB 8.1 MB/s eta 0:00:01
   ------------------------------ --------- 7.3/9.7 MB 8.2 MB/s eta 0:00:01
   ------------------------------------- -- 9.2/9.7 MB 8.4 MB/s eta 0:00:01
   ---------------------------------------- 9.7/9.7 MB 8.3 MB/s  0:00:01
Using cached numpy-2.4.4-cp313-cp313-win_amd64.whl (12.3 MB)

   ---------------------------------------- 0/3 [tzdata]
   ------------- -------------------------- 1/3 [numpy]
   ------------- -------------------------- 1/3 [numpy]
   ------------- -------------------------- 1/3 [numpy]
   ------------- -------------------------- 1/3 [nu

## Q1 - Trung vi khoang cach giua 2 lan dat hang cua cung 1 customer
**Logic SQL:** Dung `LAG(order_date)` theo tung `customer_id`, tinh so ngay chenh lech, sau do lay `MEDIAN`.

**Trap tranh duoc:** Khong lay trung binh truc tiep tren orders thang, vi can xet khoang cach theo chuoi dat hang cua moi customer.

In [4]:
query_q1 = '''
WITH ordered_orders AS (
    SELECT
        customer_id,
        CAST(order_date AS DATE) AS order_date,
        LAG(CAST(order_date AS DATE)) OVER (
            PARTITION BY customer_id
            ORDER BY CAST(order_date AS DATE)
        ) AS prev_order_date
    FROM orders
),
inter_order_gap AS (
    SELECT
        customer_id,
        date_diff('day', prev_order_date, order_date) AS gap_days
    FROM ordered_orders
    WHERE prev_order_date IS NOT NULL
)
SELECT
    'Q1' AS question_id,
    MEDIAN(gap_days) AS metric_value,
    'median_inter_order_gap_days' AS metric_name
FROM inter_order_gap;
'''
display(con.execute(query_q1).df())

,question_id,metric_value,metric_name
0,Q1,144.0,median_inter_order_gap_days


## Q2 - Phan khuc co margin trung binh cao nhat
**Logic SQL:** Margin = `(price - cogs) / price`. Group theo `segment`, lay segment co `AVG(margin)` cao nhat.

**Trap tranh duoc:** Co su dung `NULLIF(price, 0)` de tranh chia cho 0.

In [5]:
query_q2 = '''
SELECT
    'Q2' AS question_id,
    segment AS metric_value,
    AVG((price - cogs) / NULLIF(price, 0)) AS avg_margin,
    'highest_margin_segment' AS metric_name
FROM products
GROUP BY segment
ORDER BY avg_margin DESC, segment
LIMIT 1;
'''
display(con.execute(query_q2).df())

,question_id,metric_value,avg_margin,metric_name
0,Q2,Standard,0.313442,highest_margin_segment


## Q3 - Ly do tra hang pho bien nhat trong Streetwear
**Logic SQL:** Join `returns` voi `products`, loc `category = 'Streetwear'`, dem tan suat `return_reason`.

**Business insight:** Sai size la ly do tra hang rat pho bien trong fashion e-commerce.

In [6]:
query_q3 = '''
SELECT
    'Q3' AS question_id,
    r.return_reason AS metric_value,
    COUNT(*) AS frequency,
    'top_streetwear_return_reason' AS metric_name
FROM returns r
INNER JOIN products p ON r.product_id = p.product_id
WHERE lower(p.category) = 'streetwear'
GROUP BY r.return_reason
ORDER BY frequency DESC, r.return_reason
LIMIT 1;
'''
display(con.execute(query_q3).df())

,question_id,metric_value,frequency,metric_name
0,Q3,wrong_size,7626,top_streetwear_return_reason


## Q4 - Traffic source co bounce rate trung binh thap nhat
**Logic SQL:** Group theo `traffic_source`, tinh `AVG(bounce_rate)`, lay min.

**Trap tranh duoc:** Khong lay min theo ngay, ma lay trung binh toan bo period theo source de dung nghia cau hoi.

In [7]:
query_q4 = '''
SELECT
    'Q4' AS question_id,
    traffic_source AS metric_value,
    AVG(bounce_rate) AS avg_bounce_rate,
    'lowest_bounce_source' AS metric_name
FROM web_traffic
GROUP BY traffic_source
ORDER BY avg_bounce_rate ASC, traffic_source
LIMIT 1;
'''
display(con.execute(query_q4).df())

,question_id,metric_value,avg_bounce_rate,metric_name
0,Q4,email_campaign,0.004458,lowest_bounce_source


## Q5 - Ty le line item co ap dung khuyen mai
**Logic SQL:** Theo rule bat buoc, mot item duoc xem la co KM neu `promo_id IS NOT NULL OR promo_id_2 IS NOT NULL`.

**Trap tranh duoc:** Neu chi dem 1 cot `promo_id` se bi thieu du lieu khuyen mai.

In [8]:
query_q5 = '''
SELECT
    'Q5' AS question_id,
    SUM(CASE WHEN promo_id IS NOT NULL OR promo_id_2 IS NOT NULL THEN 1 ELSE 0 END) * 1.0 / COUNT(*) AS metric_value,
    SUM(CASE WHEN promo_id IS NOT NULL OR promo_id_2 IS NOT NULL THEN 1 ELSE 0 END) AS promo_applied_rows,
    COUNT(*) AS total_rows,
    'promo_item_ratio' AS metric_name
FROM order_items;
'''
display(con.execute(query_q5).df())

,question_id,metric_value,promo_applied_rows,total_rows,metric_name
0,Q5,0.386635,276316.0,714669,promo_item_ratio


## Q6 - Nhom tuoi co so don trung binh/cus cao nhat
**Logic SQL:** Buoc 1 dem so don cua tung customer. Buoc 2 moi tinh trung binh theo `age_group`.

**Trap tranh duoc:** Khong average truc tiep tren orders vi se lam sai nghia 'orders per customer'.

In [9]:
query_q6 = '''
WITH per_customer AS (
    SELECT
        c.customer_id,
        c.age_group,
        COUNT(o.order_id) AS n_orders
    FROM customers c
    LEFT JOIN orders o ON c.customer_id = o.customer_id
    WHERE c.age_group IS NOT NULL
      AND trim(c.age_group) <> ''
    GROUP BY c.customer_id, c.age_group
)
SELECT
    'Q6' AS question_id,
    age_group AS metric_value,
    AVG(n_orders) AS avg_orders_per_customer,
    'highest_avg_orders_age_group' AS metric_name
FROM per_customer
GROUP BY age_group
ORDER BY avg_orders_per_customer DESC, age_group
LIMIT 1;
'''
display(con.execute(query_q6).df())

,question_id,metric_value,avg_orders_per_customer,metric_name
0,Q6,55+,5.406851,highest_avg_orders_age_group


## Q7 - Region co doanh thu cao nhat
**Logic SQL:** Tinh line revenue = `quantity * unit_price - discount_amount`, join qua `orders` + `geography`.

**Rule bat buoc:** Loai bo don khong hop le voi `order_status NOT IN ('cancelled', 'returned')`.

**Trap tranh duoc:** Neu khong loc trang thai that bai thi doanh thu vung se bi phong dai sai lech.

In [10]:
query_q7 = '''
WITH valid_orders AS (
    SELECT order_id, zip
    FROM orders
    WHERE lower(order_status) NOT IN ('cancelled', 'returned')
),
line_revenue AS (
    SELECT
        order_id,
        quantity * unit_price - discount_amount AS revenue_line
    FROM order_items
)
SELECT
    'Q7' AS question_id,
    COALESCE(g.region, 'Unknown') AS metric_value,
    SUM(lr.revenue_line) AS total_revenue,
    'highest_revenue_region' AS metric_name
FROM valid_orders vo
INNER JOIN line_revenue lr ON vo.order_id = lr.order_id
LEFT JOIN geography g ON vo.zip = g.zip
GROUP BY COALESCE(g.region, 'Unknown')
ORDER BY total_revenue DESC, metric_value
LIMIT 1;
'''
display(con.execute(query_q7).df())

,question_id,metric_value,total_revenue,metric_name
0,Q7,East,6.220254e+09,highest_revenue_region


## Q8 - Payment method pho bien nhat trong don bi huy
**Logic SQL:** Join `orders` va `payments` de check nhat quan `payment_method`, sau do moi tim mode trong don `cancelled`.

**Data maturity point:** Chung ta khong tin 1 bang duy nhat khi co 2 nguon payment, luon cross-check truoc.

**Insight can nhan manh voi judges:** Du lieu cho thay `credit_card` moi la top cancelled method (khong phai COD).

In [11]:
query_q8 = '''
WITH base AS (
    SELECT
        o.order_id,
        lower(trim(o.order_status)) AS order_status,
        lower(trim(o.payment_method)) AS payment_from_orders,
        lower(trim(p.payment_method)) AS payment_from_payments
    FROM orders o
    INNER JOIN payments p ON o.order_id = p.order_id
),
consistency AS (
    SELECT
        COUNT(*) AS joined_rows,
        SUM(CASE WHEN payment_from_orders = payment_from_payments THEN 1 ELSE 0 END) AS matched_rows,
        SUM(CASE WHEN payment_from_orders <> payment_from_payments THEN 1 ELSE 0 END) AS mismatched_rows,
        SUM(CASE WHEN payment_from_orders = payment_from_payments THEN 1 ELSE 0 END) * 1.0 / COUNT(*) AS match_ratio
    FROM base
),
mode_cancelled AS (
    SELECT
        payment_from_orders AS payment_method,
        COUNT(*) AS cancelled_count
    FROM base
    WHERE order_status = 'cancelled'
    GROUP BY payment_from_orders
    ORDER BY cancelled_count DESC, payment_method
    LIMIT 1
)
SELECT
    'Q8' AS question_id,
    m.payment_method AS metric_value,
    m.cancelled_count,
    c.match_ratio,
    c.mismatched_rows,
    'cancelled_orders_mode_payment_method_with_cross_check' AS metric_name
FROM mode_cancelled m
CROSS JOIN consistency c;
'''
display(con.execute(query_q8).df())

,question_id,metric_value,cancelled_count,match_ratio,mismatched_rows,metric_name
0,Q8,credit_card,28452,1.0,0.0,cancelled_orders_mode_payment_method_with_cros...


## Q9 - Size co return rate cao nhat
**Rule bat buoc:** Return rate phai tinh theo **so ban ghi** (record counts), KHONG tinh theo quantity.

**Logic SQL:** `COUNT(returns.product_id) * 1.0 / COUNT(order_items.product_id)` theo tung size.

In [12]:
query_q9 = '''
SELECT
    'Q9' AS question_id,
    p.size AS metric_value,
    COUNT(r.product_id) * 1.0 / COUNT(oi.product_id) AS return_rate_by_record,
    COUNT(r.product_id) AS returned_record_count,
    COUNT(oi.product_id) AS ordered_record_count,
    'highest_return_rate_size_by_record_count' AS metric_name
FROM order_items oi
INNER JOIN products p ON oi.product_id = p.product_id
LEFT JOIN returns r
       ON oi.order_id = r.order_id
      AND oi.product_id = r.product_id
WHERE p.size IN ('S', 'M', 'L', 'XL')
GROUP BY p.size
ORDER BY return_rate_by_record DESC, p.size
LIMIT 1;
'''
display(con.execute(query_q9).df())

,question_id,metric_value,return_rate_by_record,returned_record_count,ordered_record_count,metric_name
0,Q9,S,0.056515,9723,172042,highest_return_rate_size_by_record_count


## Q10 - Ky han tra gop co gia tri payment trung binh cao nhat
**Logic SQL:** Group theo `installments`, tinh `AVG(payment_value)`, lay top 1.

**Trap tranh duoc:** Co xu huong nham sang tong payment, trong khi cau hoi yeu cau gia tri trung binh.

In [13]:
query_q10 = '''
SELECT
    'Q10' AS question_id,
    installments AS metric_value,
    AVG(payment_value) AS avg_payment_value,
    'installments_with_highest_avg_payment' AS metric_name
FROM payments
GROUP BY installments
ORDER BY avg_payment_value DESC, installments
LIMIT 1;
'''
display(con.execute(query_q10).df())

,question_id,metric_value,avg_payment_value,metric_name
0,Q10,6,24446.654403,installments_with_highest_avg_payment


## Mapping Ket Qua Sang A/B/C/D (Submission-Ready)
Cell nay tong hop metric tu Q1-Q10 va map sang dap an A/B/C/D theo key da thong nhat de dien form nhanh, minh bach va tai lap duoc.

In [14]:
# Build metric table from Q1-Q10 query outputs
query_map = {
    'Q1': query_q1,
    'Q2': query_q2,
    'Q3': query_q3,
    'Q4': query_q4,
    'Q5': query_q5,
    'Q6': query_q6,
    'Q7': query_q7,
    'Q8': query_q8,
    'Q9': query_q9,
    'Q10': query_q10,
}

metric_rows = []
for qid, query in query_map.items():
    df_q = con.execute(query).df()
    metric_value = df_q.loc[0, 'metric_value'] if 'metric_value' in df_q.columns else None
    metric_rows.append({'question_id': qid, 'metric_value': metric_value})

metrics_df = pd.DataFrame(metric_rows)

# Choice key agreed by team (updated mapping)
choice_key = {
    'Q1': {'rule': 'nearest_number', 'A': 30, 'B': 90, 'C': 180, 'D': 365},
    'Q2': {'rule': 'exact_text', 'A': 'Premium', 'B': 'Budget', 'C': 'Luxury', 'D': 'Standard'},
    'Q3': {'rule': 'exact_text', 'A': 'defective_item', 'B': 'wrong_size', 'C': 'late_delivery', 'D': 'changed_mind'},
    'Q4': {'rule': 'exact_text', 'A': 'organic_search', 'B': 'paid_search', 'C': 'email_campaign', 'D': 'social_media'},
    'Q5': {'rule': 'nearest_number', 'A': 0.19, 'B': 0.27, 'C': 0.39, 'D': 0.48},
    'Q6': {'rule': 'exact_text', 'A': '55+', 'B': '18-24', 'C': '25-34', 'D': '35-44'},
    'Q7': {'rule': 'exact_text', 'A': 'North', 'B': 'South', 'C': 'East', 'D': 'West'},
    'Q8': {'rule': 'exact_text', 'A': 'credit_card', 'B': 'cod', 'C': 'bank_transfer', 'D': 'e_wallet'},
    'Q9': {'rule': 'exact_text', 'A': 'S', 'B': 'M', 'C': 'L', 'D': 'XL'},
    'Q10': {'rule': 'nearest_number', 'A': 1, 'B': 3, 'C': 6, 'D': 12},
}

def map_choice(metric_value, cfg):
    options = {k: v for k, v in cfg.items() if k in ['A', 'B', 'C', 'D']}
    rule = cfg.get('rule', 'exact_text')

    if rule == 'nearest_number':
        metric_num = float(metric_value)
        best = sorted(
            [(opt, abs(metric_num - float(val))) for opt, val in options.items()],
            key=lambda x: (x[1], x[0])
        )[0]
        return best[0], f'nearest_number distance={best[1]:.6f}'

    metric_txt = str(metric_value).strip().lower()
    for opt, val in options.items():
        if metric_txt == str(val).strip().lower():
            return opt, 'exact_text matched'
    return 'UNMAPPED', 'no exact text match'

mapped_rows = []
for _, row in metrics_df.iterrows():
    qid = row['question_id']
    metric = row['metric_value']
    choice, reason = map_choice(metric, choice_key[qid])
    mapped_rows.append({
        'question_id': qid,
        'metric_value': metric,
        'selected_choice': choice,
        'choice_reason': reason
    })

answers_df = pd.DataFrame(mapped_rows)
submission_view = answers_df[['question_id', 'selected_choice']].copy()

print('Bang chi tiet mapping:')
display(answers_df)

print('Bang dap an de dien form:')
display(submission_view)

Bang chi tiet mapping:


,question_id,metric_value,selected_choice,choice_reason
0,Q1,144.0,C,nearest_number distance=36.000000
1,Q2,Standard,D,exact_text matched
2,Q3,wrong_size,B,exact_text matched
3,Q4,email_campaign,C,exact_text matched
4,Q5,0.386635,C,nearest_number distance=0.003365
5,Q6,55+,A,exact_text matched
6,Q7,East,C,exact_text matched
7,Q8,credit_card,A,exact_text matched
8,Q9,S,A,exact_text matched
9,Q10,6,C,nearest_number distance=0.000000


Bang dap an de dien form:


,question_id,selected_choice
0,Q1,C
1,Q2,D
2,Q3,B
3,Q4,C
4,Q5,C
5,Q6,A
6,Q7,C
7,Q8,A
8,Q9,A
9,Q10,C
